IMPLEMENTATION OF DIFFERENT ACTIVATION FUNCTIONS TO TRAIN NEURAL NETWORK

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return sigmoid(x) * (1 - sigmoid(x))

def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - np.tanh(x)**2

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return np.where(x <= 0, 0, 1)

def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

def leaky_relu_derivative(x, alpha=0.01):
    return np.where(x > 0, 1, alpha)

class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, activation='sigmoid'):
        self.weights_input_hidden = np.random.randn(input_size, hidden_size)
        self.biases_input_hidden = np.zeros(hidden_size)
        self.weights_hidden_output = np.random.randn(hidden_size, output_size)
        self.biases_hidden_output = np.zeros(output_size)
        acts = {
            'sigmoid': (sigmoid, sigmoid_derivative),
            'tanh': (tanh, tanh_derivative),
            'relu': (relu, relu_derivative),
            'leaky_relu': (leaky_relu, leaky_relu_derivative)
        }
        self.activation_function, self.activation_derivative = acts[activation]

    def feedforward(self, inputs):
        h = self.activation_function(np.dot(inputs, self.weights_input_hidden) + self.biases_input_hidden)
        return self.activation_function(np.dot(h, self.weights_hidden_output) + self.biases_hidden_output)

inputs = np.array([[0.1, 0.2], [0.3, 0.4]])

for act in ['sigmoid', 'tanh', 'relu', 'leaky_relu']:
    nn = NeuralNetwork(2, 3, 1, activation=act)
    print("Output (", act, "):", nn.feedforward(inputs))

Output ( sigmoid ): [[0.32579729]
 [0.33468455]]
Output ( tanh ): [[-0.33762433]
 [-0.58261079]]
Output ( relu ): [[0.43059566]
 [0.84380286]]
Output ( leaky_relu ): [[0.0011233 ]
 [0.00460223]]


IMPLEMENTATION OF DIFFERENT LEARNING RULES

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return sigmoid(x) * (1 - sigmoid(x))

def gradient_descent(W, b, dW, db, lr):
    return W - lr*dW, b - lr*db

def momentum(W, b, dW, db, lr, pW, pb, m=0.9):
    dW = m*pW + lr*dW
    db = m*pb + lr*db
    return W-dW, b-db, dW, db

def rmsprop(W, b, dW, db, lr, cW, cb, dr=0.9, eps=1e-8):
    cW = dr*cW + (1-dr)*dW**2
    cb = dr*cb + (1-dr)*db**2
    return W - lr*dW/(np.sqrt(cW)+eps), b - lr*db/(np.sqrt(cb)+eps), cW, cb

class NeuralNetwork:
    def __init__(self, inp, hid, out):
        self.W1 = np.random.randn(inp, hid)
        self.b1 = np.zeros(hid)
        self.W2 = np.random.randn(hid, out)
        self.b2 = np.zeros(out)

    def feedforward(self, X):
        self.h = sigmoid(X @ self.W1 + self.b1)
        return sigmoid(self.h @ self.W2 + self.b2)

    def backprop(self, X, y, lr=0.1, rule='gradient_descent'):
        out = self.feedforward(X)
        d2 = (y - out) * sigmoid_derivative(out)
        d1 = (d2 @ self.W2.T) * sigmoid_derivative(self.h)

        dW2, db2 = self.h.T @ d2, d2.sum(axis=0)
        dW1, db1 = X.T @ d1, d1.sum(axis=0)

        if rule == 'gradient_descent':
            self.W2, self.b2 = gradient_descent(self.W2, self.b2, -dW2, -db2, lr)
            self.W1, self.b1 = gradient_descent(self.W1, self.b1, -dW1, -db1, lr)

X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([[0],[1],[1],[0]], dtype=float)

nn = NeuralNetwork(2, 3, 1)

for _ in range(1000):
    nn.backprop(X, y, lr=0.1, rule='gradient_descent')

print("Final Output:")
print(nn.feedforward(X))

Final Output:
[[0.5162331 ]
 [0.48608121]
 [0.51256381]
 [0.4824081 ]]


IMPLEMENTATION OF PERCEPTRON NETWORKS

In [ ]:
import numpy as np

class Perceptron:
    def __init__(self, num_features, learning_rate=0.01, epochs=100):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = np.zeros(num_features + 1)

    def predict(self, inputs):
        x = np.insert(inputs, 0, 1)
        return 1 if np.dot(x, self.weights) > 0 else 0

    def train(self, X, y):
        for _ in range(self.epochs):
            for xi, yi in zip(X, y):
                err = yi - self.predict(xi)
                self.weights += self.lr * err * np.insert(xi, 0, 1)

X = np.array([[0,0],[0,1],[1,0],[1,1]])
y = np.array([0, 0, 0, 1])

p = Perceptron(num_features=2)
p.train(X, y)

print("Testing the Perceptron:")
for xi in X:
    print("Input:", xi, "Prediction:", p.predict(xi))

Testing the Perceptron:
Input: [0 0] Prediction: 0
Input: [0 1] Prediction: 0
Input: [1 0] Prediction: 0
Input: [1 1] Prediction: 1


PATTERN MATCHING USING DIFFERENT RULES

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([0, 1, 1, 0], dtype=float)

def create_model(rule):
    model = Sequential()
    if rule == 'single_layer':
        model.add(Dense(1, input_shape=(2,), activation='sigmoid'))
    elif rule == 'multiple_layers':
        model.add(Dense(4, input_shape=(2,), activation='relu'))
        model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

for rule in ['single_layer', 'multiple_layers']:
    print("\nTraining model with rule:", rule)
    model = create_model(rule)
    model.fit(X, y, epochs=100, verbose=0)
    loss, acc = model.evaluate(X, y, verbose=0)
    print("Test Loss:", round(loss,4), "Test Accuracy:", round(acc,4))
    print("Predictions:", model.predict(X).ravel().round(3))


Training model with rule: single_layer


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Test Loss: 0.8451 Test Accuracy: 0.75
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
Predictions: [0.476 0.74  0.684 0.872]

Training model with rule: multiple_layers
Test Loss: 0.692 Test Accuracy: 0.75
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Predictions: [0.478 0.57  0.577 0.635]


HEALTHCARE APPLICATION IN ARTIFICIAL NEURAL NETWORK

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

cols = ['age','sex','cp','trestbps','chol','fbs','restecg',
        'thalach','exang','oldpeak','slope','ca','thal','target']

data = pd.read_csv(url, names=cols).replace('?', np.nan).dropna()

data = pd.get_dummies(data, columns=['cp','restecg','slope','thal'])

X = data.drop('target', axis=1)
y = (data['target'] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)

loss, acc = model.evaluate(X_test, y_test)

print("Test Loss:", round(loss,4))
print("Test Accuracy:", round(acc,4))

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.4726 - loss: 0.8506
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4768 - loss: 0.7625
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5401 - loss: 0.6999
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5907 - loss: 0.6562
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6751 - loss: 0.6173
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7257 - loss: 0.5867 
Epoch 7/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7468 - loss: 0.5565
Epoch 8/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7848 - loss: 0.5288 
Epoch 9/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8101 - loss: 0.5030
Epoch 10/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8101 - loss: 0.4781
Epoch 11/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8143 - loss: 0.4573 
Epoch 12/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8228 - loss: 0.4359
Epoch 13/50


BUSINESS ANALYTICS APPLICATION USING ARTIFICIAL NEURAL NETWORK

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

data = pd.read_csv('/content/sales_data.csv', parse_dates=['Date'], index_col='Date')

data = data.resample('M').sum().dropna()

scaler = MinMaxScaler()
scaled = scaler.fit_transform(data)

X, y = scaled[:-1], scaled[1:]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

model = Sequential([
    LSTM(50, activation='relu', input_shape=(1, X_train.shape[2])),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')

model.fit(X_train, y_train, epochs=100, batch_size=32)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Root Mean Squared Error (RMSE):", round(rmse,2))

/tmp/ipykernel_1579/2931608917.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data = pd.read_csv('/content/sales_data.csv', parse_dates=['Date'], index_col='Date')
/tmp/ipykernel_1579/2931608917.py:11: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  data = data.resample('M').sum().dropna()
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - loss: 5.3580e-06
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 3.6189e-06
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - loss: 1.7044e-06
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - loss: 1.7458e-07
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - loss: 7.7850e-08
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - loss: 4.1600e-07
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - loss: 7.2256e-07
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 8.2819e-07
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - loss: 7.4442e-07
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 5.5250e-07
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - loss: 3.3747e-07
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - loss: 1.6032e-07
Epoch 13/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - loss: 5.7702e-08
Epoch 14/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - loss: 4.7579e-08
Epoch 15/100
1/1 ━━━━━

SPORTS ANALYTICS APPLICATION USING
ARTIFICIAL NEURAL NETWORK

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

data = pd.read_csv('sports_data.csv')

X = data.drop('outcome', axis=1)
y = data['outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)

loss, acc = model.evaluate(X_test, y_test)

print("Test Loss:", round(loss,4))
print("Test Accuracy:", round(acc,4))

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.1111 - loss: 0.7763
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.1111 - loss: 0.7635
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.1111 - loss: 0.7509
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.1111 - loss: 0.7386
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.2222 - loss: 0.7265
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.3333 - loss: 0.7148
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5556 - loss: 0.7032
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5556 - loss: 0.6918
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5556 - loss: 0.6807
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5556 - loss: 0.6697
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 1.0000 - loss: 0.6597
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 1.0000 - loss: 0.6504
Epoch 13/50
1/

TIME SERIES ANALYSIS & FORECASTING


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

data = pd.read_csv('/content/Electric_Production.csv', parse_dates=['Date'], index_col='Date')

lag = 3
for i in range(1, lag+1):
    data[f'lag_{i}'] = data['Value'].shift(i)

data.dropna(inplace=True)

X = data.drop('Value', axis=1)
y = data['Value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')

model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE:", round(rmse,2))


BACK PROPAGATION NETWORK FOR XOR
FUNCTION


In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([[0],[1],[1],[0]], dtype=float)

np.random.seed(1)

W1 = 2*np.random.random((2, 3)) - 1
W2 = 2*np.random.random((3, 1)) - 1

lr = 0.1

for epoch in range(10000):
    h = sigmoid(X @ W1)
    o = sigmoid(h @ W2)

    d2 = (y - o) * sigmoid_derivative(o)
    d1 = (d2 @ W2.T) * sigmoid_derivative(h)

    W2 += h.T @ d2 * lr
    W1 += X.T @ d1 * lr

print("Output after training:")
print(np.round(sigmoid(sigmoid(X @ W1) @ W2),4))

Output after training:
[[0.0908]
 [0.9111]
 [0.9121]
 [0.5114]]


IMPLEMENTATION OF SINGLE LAYER PERCEPTRON FOR OR GATE

In [ ]:
import numpy as np

class Perceptron:
    def __init__(self, lr=0.1, epochs=20):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = 0.0

    def activation(self, z):
        return 1 if z >= 0 else 0

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        for _ in range(self.epochs):
            for xi, yi in zip(X, y):
                pred = self.activation(np.dot(self.w, xi) + self.b)
                err = yi - pred
                self.w += self.lr * err * xi
                self.b += self.lr * err

    def predict(self, X):
        return [self.activation(np.dot(self.w, xi) + self.b) for xi in X]

X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([0, 1, 1, 1])

p = Perceptron(lr=0.1, epochs=20)
p.fit(X, y)

preds = p.predict(X)

print("OR Gate Predictions:")
for xi, pi in zip(X, preds):
    print(xi.astype(int), "->", pi)

OR Gate Predictions:
[0 0] -> 0
[0 1] -> 1
[1 0] -> 1
[1 1] -> 1


IMPLEMENTATION OF HEBBIAN LEARNING RULE

In [ ]:
import numpy as np

X = np.array([[ 1, 1, -1, -1],
              [ 1, -1, 1, -1],
              [-1, 1, 1, -1]], dtype=float)

y = np.array([1, -1, 1, -1], dtype=float)

n_inputs = X.shape[1]
n_patterns = X.shape[0]

W = np.zeros(n_inputs)
b = 0.0

for xi, yi in zip(X, y):
    W += xi * yi
    b += yi

W /= n_patterns
b /= n_patterns

print("Hebbian Weights:", np.round(W,4))
print("Bias:", round(b,4))
print()

for xi, yi in zip(X, y):
    out = np.sign(np.dot(W, xi) + b)
    print("Input:", xi, "Target:", yi, "Output:", out)

Hebbian Weights: [-0.3333  1.     -0.3333 -0.3333]
Bias: 0.3333

Input: [ 1.  1. -1. -1.] Target: 1.0 Output: 1.0
Input: [ 1. -1.  1. -1.] Target: -1.0 Output: -1.0
Input: [-1.  1.  1. -1.] Target: 1.0 Output: 1.0


IMPLEMENTATION OF DELTA LEARNING RULE (WIDROW-HOFF)

In [ ]:
import numpy as np

def linear(x):
    return x

def mse(y, t):
    return np.mean((y - t)**2)

X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([0, 1, 1, 1], dtype=float)

W = np.zeros(2)
b = 0.0
lr = 0.1

print("Training with Delta Rule:")

for epoch in range(50):
    total_loss = 0
    for xi, yi in zip(X, y):
        pred = np.dot(W, xi) + b
        err = yi - pred
        W += lr * err * xi
        b += lr * err
        total_loss += err**2
    if epoch % 10 == 0:
        print("Epoch", epoch, "MSE =", round(total_loss/4,5))

print()
print("Final Weights:", np.round(W,4), "Bias:", round(b,4))
print("Predictions:", [round(np.dot(W, xi)+b,3) for xi in X])

Training with Delta Rule:
Epoch 0 MSE = 0.5486
Epoch 10 MSE = 0.08339
Epoch 20 MSE = 0.07897
Epoch 30 MSE = 0.07773
Epoch 40 MSE = 0.07736

Final Weights: [0.4388 0.4663] Bias: 0.2856
Predictions: [np.float64(0.286), np.float64(0.752), np.float64(0.724), np.float64(1.191)]


IMPLEMENTATION OF GRADIENT DESCENT OPTIMIZATION

In [ ]:
import numpy as np

f = lambda w: (w - 3)**2
grad_f = lambda w: 2*(w - 3)

print("Gradient Descent with lr=0.1")

w = 0.0
lr = 0.1

for t in range(1, 21):
    g = grad_f(w)
    w -= lr * g
    if t % 4 == 0:
        print("Step", t, "w =", round(w,5), "f(w) =", round(f(w),6))

print()

print("Different learning rates:")

for lr in [0.01, 0.1, 0.4, 0.9]:
    w = 0.0
    for _ in range(20):
        w -= lr * grad_f(w)
    print("lr =", lr, "final w =", round(w,5), "f(w) =", round(f(w),6))

Gradient Descent with lr=0.1
Step 4 w = 1.7712 f(w) = 1.509949
Step 8 w = 2.49668 f(w) = 0.253327
Step 12 w = 2.79384 f(w) = 0.042501
Step 16 w = 2.91556 f(w) = 0.007131
Step 20 w = 2.96541 f(w) = 0.001196

Different learning rates:
lr = 0.01 final w = 0.99718 f(w) = 4.011304
lr = 0.1 final w = 2.96541 f(w) = 0.001196
lr = 0.4 final w = 3.0 f(w) = 0.0
lr = 0.9 final w = 2.96541 f(w) = 0.001196


IMPLEMENTATION OF MOMENTUM-BASED GRADIENT DESCENT

In [ ]:
import numpy as np

f = lambda w: (w - 5)**2
grad_f = lambda w: 2*(w - 5)

lr = 0.1
beta = 0.9
steps = 30

w_gd = 0.0
gd_path = [w_gd]

for _ in range(steps):
    w_gd -= lr * grad_f(w_gd)
    gd_path.append(w_gd)

w_m = 0.0
v = 0.0
mom_path = [w_m]

for _ in range(steps):
    v = beta * v + lr * grad_f(w_m)
    w_m -= v
    mom_path.append(w_m)

print("Step   Plain_GD   Momentum")

for i in range(0, steps+1, 5):
    print(i, "   ", round(gd_path[i],5), "   ", round(mom_path[i],5))

print()
print("Final Plain GD w =", round(gd_path[-1],5))
print("Final Momentum w =", round(mom_path[-1],5))

Step   Plain_GD   Momentum
0     0.0     0.0
5     3.3616     7.9021
10     4.46313     4.978
15     4.82408     3.30889
20     4.94235     6.74626
25     4.98111     4.20879
30     4.99381     4.77979

Final Plain GD w = 4.99381
Final Momentum w = 4.77979



IMPLEMENTATION OF ADAM OPTIMIZER


In [ ]:
import numpy as np

f = lambda w: w**2
grad_f = lambda w: 2*w

lr = 0.1
b1 = 0.9
b2 = 0.999
eps = 1e-8

w = 10.0
m = 0.0
v = 0.0

print("Adam Optimizer")

for t in range(1, 31):
    g = grad_f(w)
    m = b1*m + (1-b1)*g
    v = b2*v + (1-b2)*g**2
    m_hat = m / (1 - b1**t)
    v_hat = v / (1 - b2**t)
    w -= lr * m_hat / (np.sqrt(v_hat) + eps)
    if t % 5 == 0:
        print("Step", t, "w =", round(w,5), "f(w) =", round(f(w),6))

print()
print("Final w =", round(w,5))

Adam Optimizer
Step 5 w = 9.50046 f(w) = 90.258771
Step 10 w = 9.0035 f(w) = 81.06295
Step 15 w = 8.51147 f(w) = 72.44515
Step 20 w = 8.02666 f(w) = 64.427342
Step 25 w = 7.55116 f(w) = 57.020088
Step 30 w = 7.08681 f(w) = 50.222918

Final w = 7.08681
